# CourtVision: train, benchmark, evaluate

Run this on **Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU). It produces the three numbers for the README and resume:

1. **mAP@0.5** of a fine-tuned ball + player detector
2. **End-to-end FPS** on a T4
3. **False-positive possession events**, pose vs. box-only, on hand-labelled clips

Every number comes from your own run. Paste them into the README's Results tables.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Lahanas/Real-Time-Basketball-Analytics-Pipeline.git
%cd Real-Time-Basketball-Analytics-Pipeline
!pip install -q -e . roboflow

## 1. Dataset

Use a basketball detection dataset from [Roboflow Universe](https://universe.roboflow.com) with at least a **ball** class and a **player** class. The one linked in the Code In a Jiffy tutorial's description works.

On the dataset page: **Download Dataset → YOLOv8 → show download code**, then copy the workspace, project and version into the cell below. Your API key is in Roboflow → Settings → API Keys.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = ""   # e.g. the part after universe.roboflow.com/
PROJECT   = ""
VERSION   = 1

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location="datasets/basketball")
DATA_YAML = f"{dataset.location}/data.yaml"

In [ ]:
import yaml
names = yaml.safe_load(open(DATA_YAML))["names"]
names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
print(names)

def find(word):
    hits = [i for i, n in names.items() if word in n.lower()]
    assert hits, f"no class containing '{word}' - set it by hand"
    return hits[0]

BALL_CLASS, PLAYER_CLASS = find("ball"), find("player")
print("ball class:", BALL_CLASS, "| player class:", PLAYER_CLASS)

## 2. Fine-tune YOLOv8

One model for both classes. `imgsz=1280` because broadcast balls are only a few pixels wide; ~100 epochs takes roughly 1 to 2 hours on a T4. Lower `EPOCHS` for a first try.

In [ ]:
from ultralytics import YOLO

EPOCHS, IMGSZ = 100, 1280
model = YOLO("yolov8s.pt")
model.train(data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, batch=8, patience=20,
            project="runs", name="ball_player", exist_ok=True)
BEST = "runs/ball_player/weights/best.pt"

In [ ]:
import json
metrics = YOLO(BEST).val(data=DATA_YAML, imgsz=IMGSZ, split="test" if "test" in yaml.safe_load(open(DATA_YAML)) else "val")
per_class = {names[c]: round(float(metrics.box.ap50[i]), 3) for i, c in enumerate(metrics.box.ap_class_index)}
summary = {"mAP50_all": round(float(metrics.box.map50), 3), "mAP50_95_all": round(float(metrics.box.map), 3), "mAP50_per_class": per_class}
print(json.dumps(summary, indent=2))
json.dump(summary, open("detector_metrics.json", "w"), indent=2)

In [ ]:
# Point the pipeline at the fine-tuned model
import os, shutil
os.makedirs("models", exist_ok=True)
shutil.copy(BEST, "models/ball_player.pt")
cfg = {
    "models": {
        "player_detector": "models/ball_player.pt", "player_class": PLAYER_CLASS,
        "ball_model": "models/ball_player.pt", "ball_class": BALL_CLASS,
        "ball_conf": 0.10, "ball_imgsz": IMGSZ,
        "imgsz": IMGSZ, "device": 0, "half": True,
    }
}
yaml.safe_dump(cfg, open("configs/finetuned.yaml", "w"), sort_keys=False)
print(open("configs/finetuned.yaml").read())

## 3. Run + benchmark on a T4

Upload your own clip, or use one of the sample clips from the tutorial's repo. `--benchmark` disables the cache so every model really runs, and synchronises the GPU before timing each stage.

In [ ]:
VIDEO = "clip.mp4"
!wget -q -O {VIDEO} https://github.com/abdullahtarek/basketball_analysis/raw/main/input_videos/video_1.mp4
# Or: from google.colab import files; VIDEO = list(files.upload())[0]

In [ ]:
!courtvision {VIDEO} -c configs/finetuned.yaml -o outputs --benchmark --dump-possession 2>&1 | grep -v "detecting"

In [ ]:
# Compare against stock weights (no fine-tuning) on the same clip
!courtvision {VIDEO} -o outputs_stock --benchmark --dump-possession 2>&1 | grep -v "detecting"

In [ ]:
from google.colab import files
stem = os.path.splitext(os.path.basename(VIDEO))[0]
files.download(f"outputs/{stem}_annotated.mp4")

## 4. Evaluate possession

Watch `outputs/<clip>_annotated.mp4`. Frame numbers are bottom-left and track IDs sit under each player. Write down who holds the ball, as `start_frame,end_frame,player_id`. Frames not covered mean nobody has it (pass in the air, shot, loose ball). Track IDs come from this run, so label using the IDs you see in this video.

More clips → a more trustworthy number. Aim for at least 3 clips / 30+ possessions.

In [ ]:
LABELS = """start_frame,end_frame,player_id
"""  # fill in, one row per possession
open(f"outputs/{stem}_labels.csv", "w").write(LABELS)
!python evaluation/evaluate_possession.py outputs/{stem}_possession.json outputs/{stem}_labels.csv

## 5. Keep the results

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/courtvision
!cp -r models outputs detector_metrics.json configs/finetuned.yaml runs/ball_player/results.png /content/drive/MyDrive/courtvision/ 2>/dev/null; ls /content/drive/MyDrive/courtvision

## 6. Record the numbers

Copy into the README's **Results** section:

- `detector_metrics.json` → mAP@0.5 (overall and ball)
- the `Throughput:` line from step 3 → FPS on a T4, plus the stock-weights FPS for comparison
- the evaluation table from step 4 → frame accuracy and false-positive events, pose vs. box-only

Commit `configs/finetuned.yaml` and the README. Keep `.pt` weights out of git and link them (e.g. a GitHub Release) instead.